# RealVisXL V4.0 Uncensored Studio (OOM Fixed)
Uses model CPU offload and attention slicing to prevent CUDA Out of Memory errors.

In [ ]:
!pip install -q diffusers transformers accelerate gradio torch torchvision

import torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
import gradio as gr

print("Loading RealVisXL V4.0 with memory optimization...")
model_id = "SG161222/RealVisXL_V4.0"

pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

def generate_or_edit(image, prompt, negative_prompt, mode, strength, steps):
    torch.cuda.empty_cache()
    if mode == "Text to Image":
        img = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=int(steps),
            guidance_scale=7.5
        ).images[0]
    else:
        if image is None:
            return None
        # Convert txt2img pipe to img2img behavior or load img2img with offload
        from diffusers import StableDiffusionXLImg2ImgPipeline
        pipe_i2i = StableDiffusionXLImg2ImgPipeline.from_pipe(pipe)
        img = pipe_i2i(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=image,
            strength=float(strength),
            num_inference_steps=int(steps),
            guidance_scale=7.5
        ).images[0]
    return img

demo = gr.Interface(
    fn=generate_or_edit,
    inputs=[
        gr.Image(type="pil", label="Source Image for Edit"),
        gr.Textbox(label="Prompt", value="Cinematic photorealistic portrait, highly detailed"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality, deformed"),
        gr.Radio(["Text to Image", "Image to Image"], value="Text to Image", label="Mode"),
        gr.Slider(0.1, 0.9, value=0.5, label="Strength"),
        gr.Slider(10, 50, value=25, step=1, label="Steps")
    ],
    outputs=gr.Image(label="Output"),
    title="RealVisXL V4.0 Studio (OOM Fixed)"
)
demo.launch(share=True, debug=True)